In [3]:

from pathlib import Path
import pandas as pd
from scipy.stats import spearmanr

backbone, N, seeds = "georsclip-14", 10, [1, 2, 3]
metric = "mAP50"  # or "mAP50-95"
root = Path("/home/gridsan/manderson/ovdsat")
eval_dir = root / "run/eval/detection/mar"

def load_ap(path):
    col = -2 if metric == "mAP50" else -1
    return pd.Series({
        p[0]: float(p[col])
        for line in Path(path).read_text().splitlines()[1:]
        if len(p := line.split()) == 7 and p[0] != "all"
    })

# Per-class AP gain, averaged across matched runs
gains = []
for seed in seeds:
    coop = load_ap(eval_dir / f"coop_prototypes/backbone_{backbone}_boxes/N{N}-{seed}/results_{backbone}.txt")
    text = load_ap(eval_dir / f"text_prototypes/backbone_{backbone}_boxes/zeroshot_{seed}/results_{backbone}.txt")
    gains.append(coop - text)
ap_gain = pd.concat(gains, axis=1).mean(axis=1)

# Per-class cosine similarity delta
path = root / f"run/embeddings/test_cosine_delta_mar_{backbone}_N{N}-200samples.txt"
delta = {}
for line in path.read_text().splitlines():
    p = line.split()
    if len(p) == 12 and p[2] == p[5] == p[8] == "+/-":
        delta[p[0]] = float(p[7])

# Correlation across 20 classes
df = pd.DataFrame({
    "delta_cosim": pd.Series(delta),
    "ap_gain": ap_gain
}).dropna().sort_index()

assert len(df) == 20, f"Expected 20 classes, found {len(df)}"

rho, p = spearmanr(df["delta_cosim"], df["ap_gain"])
print(df.to_string())
print(f"\nSpearman ρ = {rho:.3f}, p = {p:.4g}")


        delta_cosim   ap_gain
B-1B         0.0264 -0.024167
B-52         0.0194  0.159867
C-130        0.0172  0.176100
C-17         0.0125  0.121467
C-5          0.0150  0.195260
E-3          0.0302  0.719667
E-8          0.0170  0.199867
F-15         0.0241  0.401733
F-16         0.0161  0.103267
F-22         0.0128 -0.103357
FA-18        0.0317  0.779367
KC-1         0.0371  0.292967
KC-135       0.0215  0.292233
P-3C         0.0079  0.376733
SU-24        0.0066  0.035170
SU-34        0.0171  0.198767
SU-35       -0.0006  0.135400
TU-160       0.0174  0.412433
TU-22        0.0113  0.285933
TU-95       -0.0064  0.241933

Spearman ρ = 0.433, p = 0.05647
